In [9]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from scipy.spatial import ConvexHull
from pymatgen.core.composition import Composition
from pymatgen.core.lattice import Lattice
from pymatgen.core.composition import Composition
from ase.io import read


# Constants and Settings.
E_Na_ref = -1.31 # From materials project.
F = 96485.33212  # Faraday constant (C/mol)
# EXCLUDE_ORDERINGS = {"JLK_c"} # Wierd outlier in convex hull.
EXCLUDE_ORDERINGS = {}
PHASE_CHANGE_EDGE_WIDTH = 0.02
P3_LINESTYLE = (0, (2, 1))  # (offset, (on, off, ...)) -> nicer than default "dotted"

# Plot Settings.
plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
    "legend.title_fontsize": 15,
})


def theoretical_capacity_full_mAh_g(tm: str) -> float:
    """
    Theoretical max capacity for 1 Na per TMO2 (i.e., NaTMO2),
    in mAh/g, using formula mass of NaTMO2.
    """
    comp = Composition(f"Na1{tm}1O2")
    M = comp.weight  # g/mol
    n = 1.0          # 1 e- per Na
    return n * F / 3.6 / M


def energy_per_TMO2(row):
    """Return energy per TMO2 unit in eV."""
    formula = row["formula"]
    tm_symbol = row["tm"]
    natoms = row["natoms"]

    comp_norm = Composition(formula)
    n_norm = comp_norm.num_atoms  # e.g. 22 for Na(CoO2)7

    if n_norm == 0 or natoms == 0:
        return np.nan

    scale = natoms / n_norm
    n_TMO2_units = comp_norm[tm_symbol] * scale

    if n_TMO2_units == 0:
        return np.nan

    if "energy_eV" in row and not pd.isna(row["energy_eV"]):
        E_tot = row["energy_eV"]
    elif "energy_per_atom_eV" in row and not pd.isna(row["energy_per_atom_eV"]):
        E_tot = row["energy_per_atom_eV"] * natoms
    else:
        return np.nan

    return E_tot / n_TMO2_units


def lower_hull_indices_scipy(xs, ys):
    """
    Compute indices of points on the lower convex hull (in energy)
    using scipy.spatial.ConvexHull.

    - xs, ys: 1D arrays of the same length
    - Returns indices into the original arrays
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs)
    if n == 0:
        return []
    # Collapse multiple points at the same x to the lowest-energy one
    unique_indices = []
    for x0 in sorted(set(xs)):
        maskx = np.isclose(xs, x0)
        idxs = np.where(maskx)[0]
        j = idxs[np.argmin(ys[idxs])]
        unique_indices.append(j)

    xs_u = xs[unique_indices]
    ys_u = ys[unique_indices]

    if len(xs_u) <= 2:
        # Everything is on the "hull"
        return sorted(unique_indices, key=lambda i: xs[i])

    pts = np.column_stack([xs_u, ys_u])
    hull = ConvexHull(pts)
    verts = hull.vertices  # indices into xs_u / ys_u

    # Sort hull vertices by x, then y
    verts_sorted = sorted(verts, key=lambda i: (xs_u[i], ys_u[i]))

    # One pass to keep only the lower chain
    lower_u = []
    for i in verts_sorted:
        while len(lower_u) >= 2:
            j = lower_u[-2]
            k = lower_u[-1]
            x1, y1 = xs_u[j], ys_u[j]
            x2, y2 = xs_u[k], ys_u[k]
            x3, y3 = xs_u[i], ys_u[i]
            cross = (x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)
            if cross <= 0:
                # New point is below or collinear in an "upward" sense:
                # remove the previous point from the lower hull
                lower_u.pop()
            else:
                break
        lower_u.append(i)

    # Map back to original indices
    lower_idx = [unique_indices[i] for i in lower_u]
    return sorted(lower_idx, key=lambda i: xs[i])


def lattice_c_from_cell(cell):
    """
    Extract c lattice parameter (Å) from CSV 'lattice' field.

    The field was written as final_struct.lattice.as_dict(),
    so we parse the dict and reconstruct a pymatgen Lattice.
    """
    if cell is None:
        return np.nan
    if isinstance(cell, float) and np.isnan(cell):
        return np.nan

    try:
        # If already a dict
        if isinstance(cell, dict):
            d = cell
        else:
            # Most CSV cases: stringified python dict with single quotes + tuples
            d = ast.literal_eval(str(cell))

        lat = Lattice.from_dict(d)
        return float(lat.c)

    except Exception:
        # Fallback: try extracting from matrix directly
        try:
            mat = np.array(d.get("matrix", None), dtype=float)
            if mat.shape == (3, 3):
                return float(np.linalg.norm(mat[2]))
        except Exception:
            pass

    return np.nan


def short_name(ordering: str) -> str:
    return ordering.replace("_w_vac", "")


def phase_kind(phase_str: str) -> str:
    """Classify phase string into 'O3', 'P3', 'O1', or 'other'."""
    s = str(phase_str)
    if "O3" in s:
        return "O3"
    if "P3" in s:
        return "P3"
    if "O1" in s:
        return "O1"
    return "other"


# Load DFT summary (from dft_collect_from_single_TM.py)
# df = pd.read_csv("../data/singleTM_w_exp_2nd_relax.csv")  # adjust path as needed
df = pd.read_csv("../data/singleTM_tighter_conv.csv")

# Keep only clean, converged runs.
mask = pd.Series(True, index=df.index)
if "status" in df.columns:
    mask &= df["status"] == "OK"
if "n_custodian_errors" in df.columns:
    mask &= df["n_custodian_errors"] == 0
if "converged" in df.columns:
    mask &= df["converged"] == True

df = df[mask].copy()
df["E_TMO2_eV"] = df.apply(energy_per_TMO2, axis=1)
df["lattice_c_A"] = df["lattice"].apply(lattice_c_from_cell)
df = df.dropna(subset=["E_TMO2_eV"]).copy()

# Map ordering -> x_Na by hard coded map.
na_map = {
    "AJT_a": 1/7, "AJT_b": 1/4, "AJT_c": 1/4, "AJT_d": 1/3, "AJT_e": 1/3, 
    "AJT_f": 2/5, "AJT_g": 1/2, "AJT_h": 3/5, "AJT_i": 5/8, "AJT_j": 5/8,
    "AJT_k": 2/3, "AJT_l": 2/3, "AJT_m": 2/3, "AJT_n": 3/4, "AJT_o": 3/4,
    "AJT_p": 5/6, "AJT_q": 7/8, "AJT_r": 8/9,
    "JLK_b": 3/8, "JLK_c": 7/10, "JLK_theta_4": 4/5, "JLK_theta_5": 5/6,
    "JLK_delta": 2/3, "JLK_eta": 4/7, "JLK_zeta": 1/2, "JLK_zeta2": 2/5,
    "JLK_zeta3": 3/7, "JLK_zeta6": 6/11,
    "CD_a": 1/2, "JMAS_a": 1/3, "JMAS_b": 2/5, "JMAS_e": 1, "XC_a": 1/3,
    "XC_b": 1/2, "JMAS_c": 1/2, "JMAS_d": 2/3, "XL_a": 5/8
}

# Prefer meta_ordering if available, else fallback to ordering
ordering_raw = df["meta_ordering"].fillna(df["ordering"])
df["ordering_short"] = ordering_raw.apply(short_name)
if EXCLUDE_ORDERINGS:
    df = df[~df["ordering_short"].isin(EXCLUDE_ORDERINGS)].copy()

# First map all the enumerated orderings
df["xNa"] = df["ordering_short"].map(na_map)

# Override for endmembers:
mask_full = df["ordering"] == "full"
mask_empty = df["ordering"] == "empty"
df.loc[mask_full, "xNa"] = 1.0
df.loc[mask_empty, "xNa"] = 0.0

# Drop anything that still doesn't have an xNa
df = df.dropna(subset=["xNa"]).copy()

# Label with the chosen group (from metadata if available)
if "meta_group_min" in df.columns:
    df["group_label"] = df["meta_group_min"].fillna("")
else:
    df["group_label"] = ""



In [8]:
# Plotting convex hulls per TM
tms = sorted(df["tm"].unique())
out_paths = []

for tm in tms:
    df_tm = df[df["tm"] == tm].copy()
    if df_tm.empty:
        continue

    # One point per (ordering_short, phase, xNa)
    df_pts = (
        df_tm
        .groupby(["ordering_short", "phase", "xNa"], as_index=False)
        .agg({
            "E_TMO2_eV": "mean",
            "group_label": "first",
            "lattice_c_A": "mean",
        })
    )

    if df_pts.empty:
        continue

    xs = df_pts["xNa"].to_numpy()
    ys = df_pts["E_TMO2_eV"].to_numpy()

    # Convex hull indices on *raw* energies (affine shift won't change hull)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    hull_mask = np.zeros(len(df_pts), dtype=bool)
    hull_mask[hull_idx] = True    
    
    # -------------------------------------------------
    # Voltage curve from convex hull (per TMO2)
    # -------------------------------------------------
    xs_arr = df_pts["xNa"].to_numpy()
    E_arr = df_pts["E_TMO2_eV"].to_numpy()

    # Hull points sorted by composition
    hull_idx_sorted = sorted(hull_idx, key=lambda i: xs_arr[i])
    x_hull = np.array([xs_arr[i] for i in hull_idx_sorted], dtype=float)
    E_hull = np.array([E_arr[i] for i in hull_idx_sorted], dtype=float)
    c_arr = df_pts["lattice_c_A"].to_numpy()
    c_hull = np.array([c_arr[i] for i in hull_idx_sorted], dtype=float)

    # Compute plateau voltages between neighboring hull points
    V_segments = []  # list of (x1, x2, V)
    for i in range(len(x_hull) - 1):
        x1, x2 = x_hull[i], x_hull[i + 1]
        if np.isclose(x1, x2):
            continue  # avoid division by zero if something weird slips in

        # Na chemical potential in host from hull segment (eV per Na)
        mu_Na_host = (E_hull[i + 1] - E_hull[i]) / (x2 - x1)

        # Voltage vs Na/Na+ (V)
        V_seg = - (mu_Na_host - E_Na_ref)

        V_segments.append((x1, x2, V_seg))

    # -------------------------------------------------
    # Plot voltage vs x as step-like plateaus
    # -------------------------------------------------
    figV, axV = plt.subplots(figsize=(6, 4.5))
    p3_spans = []         # P3–P3 plateaus for solid red shading
    gradient_spans = []   # O3–P3 or P3–O3 plateaus for white→red gradient

    # Horizontal plateaus, styled by endpoint phases
    for i, (x1, x2, V_seg) in enumerate(V_segments):
        idx_left = hull_idx_sorted[i]
        idx_right = hull_idx_sorted[i + 1]

        phase_left = df_pts.loc[idx_left, "phase"]
        phase_right = df_pts.loc[idx_right, "phase"]

        kind_left = phase_kind(phase_left)
        kind_right = phase_kind(phase_right)

        x_mid = 0.5 * (x1 + x2)

        # Decide line style based on endpoint phases
        if kind_left == "O3" and kind_right == "O3":
            # O3–O3: solid
            axV.hlines(
                V_seg, x1, x2,
                linewidth=3.0,
                color="black",
                linestyles="solid",
            )
        elif kind_left == "P3" and kind_right == "P3":
            # P3–P3: dotted
            axV.hlines(
                V_seg, x1, x2,
                linewidth=3.0,
                color="black",
                linestyles=P3_LINESTYLE,
            )
            p3_spans.append((x1, x2))
        elif (
            (kind_left == "O3" and kind_right == "P3") or
            (kind_left == "P3" and kind_right == "O3")
        ):
            # O3–P3: half solid, half dotted
            # Left half uses left phase style, right half uses right phase style
            if kind_left == "O3" and kind_right == "P3":
                left_style, right_style = "solid", P3_LINESTYLE
                p3_on_right = True
            else:
                left_style, right_style = P3_LINESTYLE, "solid"
                p3_on_right = False

            axV.hlines(
                V_seg, x1, x_mid,
                linewidth=3.0,
                color="black",
                linestyles=left_style,
            )
            axV.hlines(
                V_seg, x_mid, x2,
                linewidth=3.0,
                color="black",
                linestyles=right_style,
            )
            gradient_spans.append((x1, x2, p3_on_right))
        else:
            # Any segments involving O1 or 'other' → default to solid
            axV.hlines(
                V_seg, x1, x2,
                linewidth=3.0,
                color="black",
                linestyles="solid",
            )

        # Voltage label at plateau center (keep as before)
        axV.text(
            x_mid,
            V_seg + 0.06,
            f"{V_seg:.2f} V",
            ha="center",
            va="bottom",
            fontsize=6,
        )

    # 2) Vertical lines connecting neighboring plateaus at hull compositions
    if len(x_hull) >= 2 and len(V_segments) == len(x_hull) - 1:
        # internal hull points only (endpoints have no vertical jump)
        for j in range(1, len(x_hull) - 1):
            x_boundary = x_hull[j]
            V_left = V_segments[j - 1][2]
            V_right = V_segments[j][2]

            # Decide vertical style based on the phase at this hull composition
            phase_mid = df_pts.loc[hull_idx_sorted[j], "phase"]
            kind_mid = phase_kind(phase_mid)

            if kind_mid == "P3":
                vstyle = P3_LINESTYLE   # fancy dotted vertical between P3 plateaus
            else:
                vstyle = "solid"

            axV.vlines(
                x_boundary,
                min(V_left, V_right),
                max(V_left, V_right),
                linewidth=3.0,
                color="black",
                linestyles=vstyle,
            )


        # 3) Markers at each hull composition on both adjacent plateaus
        for local_i, global_idx in enumerate(hull_idx_sorted):
            x_point = x_hull[local_i]

            # phase-dependent marker shape
            phase_i = df_pts.loc[global_idx, "phase"]
            if "O3" in phase_i:
                marker = "o"
            elif "P3" in phase_i:
                marker = "s"
            elif "O1" in phase_i:
                marker = "^"
            else:
                marker = "x"

            # left-side plateau (exists for all except the first hull point)
            if local_i > 0:
                V_left = V_segments[local_i - 1][2]
                axV.scatter(
                    x_point,
                    V_left,
                    marker=marker,
                    s=130,
                    facecolors="none",
                    edgecolors="black",
                    linewidths=1.7,
                    zorder=5,
                )

            # right-side plateau (exists for all except the last hull point)
            if local_i < len(V_segments):
                V_right = V_segments[local_i][2]
                axV.scatter(
                    x_point,
                    V_right,
                    marker=marker,
                    s=130,
                    facecolors="none",
                    edgecolors="black",
                    linewidths=1.7,
                    zorder=5,
                )


    axV.set_xlabel("x in Na$_x$TMO$_2$")
    axV.set_ylabel("Voltage vs Na/Na$^+$ (V)")
    # axV.set_title(f"Voltage-Capacity: TM = {tm}")

    axV.set_xlim(0, 1)
    axV.set_ylim(-0.02, 6.52)

    # ---- Top axis: capacity, manually mapped and reversed ----
    C_full = theoretical_capacity_full_mAh_g(tm)  # mAh/g at x=1

    # Create an independent top x-axis
    axTop = axV.twiny()

    # Reverse ONLY the top axis direction
    x_min, x_max = axV.get_xlim()
    axTop.set_xlim(x_max, x_min)

    axTop.set_xlabel("Theoretical capacity (mAh/g)")

    # ---- Custom capacity ticks with your rule ----
    step = 50.0
    max_cap = float(C_full)

    last_regular = (max_cap // step) * step
    cap_ticks = list(np.arange(0.0, last_regular + 0.1, step))

    gap = max_cap - last_regular
    too_close_frac = 0.30  # your “skip if too close” rule

    if cap_ticks and last_regular > 0 and gap > 0 and gap < too_close_frac * step:
        cap_ticks = [t for t in cap_ticks if abs(t - last_regular) > 1e-6]

    # Always include the true max
    cap_ticks.append(max_cap)
    cap_ticks = sorted(set(cap_ticks))

    # Map capacity ticks -> x positions
    x_positions = [t / C_full for t in cap_ticks]

    # Keep only ticks that fall within the axis range
    x_positions_filtered = []
    cap_ticks_filtered = []
    for xp, ct in zip(x_positions, cap_ticks):
        if (xp >= x_min - 1e-6) and (xp <= x_max + 1e-6):
            x_positions_filtered.append(xp)
            cap_ticks_filtered.append(ct)

    axTop.set_xticks(x_positions_filtered)
    axTop.set_xticklabels([f"{t:.0f}" for t in cap_ticks_filtered])

    """
    if V_segments:
        Vs = np.array([v for (_, _, v) in V_segments])
        V_min = float(Vs.min())
        V_max = float(Vs.max())
        pad = 0.15 * (V_max - V_min if V_max > V_min else 1.0)
        axV.set_ylim(V_min - pad, V_max + pad)
    else:
        axV.set_ylim(0.0, 4.5)  # fallback
    """

    # Volume plot
    # Secondary axis for c lattice parameter
    axC = axV.twinx()

    # Hull indices sorted by x
    hull_idx_sorted = sorted(hull_idx, key=lambda i: xs_arr[i])

    # Hull arrays
    x_hull = np.array([xs_arr[i] for i in hull_idx_sorted], dtype=float)
    c_hull = np.array([c_arr[i] for i in hull_idx_sorted], dtype=float)

    # Build phase list ONLY for hull points (length must match x_hull/c_hull)
    phase_hull = [str(df_pts.loc[i, "phase"]) for i in hull_idx_sorted]

    # Masks with correct length
    mask_o3 = np.array(["O3" in p for p in phase_hull])
    mask_p3 = np.array(["P3" in p for p in phase_hull])
    mask_o1 = np.array(["O1" in p for p in phase_hull])

    # line data
    c_hull_line = c_hull.copy()
    c_hull_line[mask_o1] *= 3

    # plot line
    axC.plot(
        x_hull, c_hull_line,
        linestyle="-",
        linewidth=1.8,
        color="red",
        zorder=3,
    )

    # Masks with correct length
    mask_o3 = np.array(["O3" in p for p in phase_hull])
    mask_p3 = np.array(["P3" in p for p in phase_hull])
    mask_o1 = np.array(["O1" in p for p in phase_hull])


    # --- Phase-dependent markers at hull points (match voltage shapes) ---
    phase_hull = [df_pts.loc[idx, "phase"] for idx in hull_idx_sorted]
    phase_s = pd.Series(phase_hull)

    mask_o3 = phase_s.str.contains("O3", na=False).to_numpy()
    mask_p3 = phase_s.str.contains("P3", na=False).to_numpy()
    mask_o1 = phase_s.str.contains("O1", na=False).to_numpy()

    # O3 hull markers: red circles
    if mask_o3.any():
        axC.scatter(
            x_hull[mask_o3], c_hull[mask_o3],
            marker="o",
            s=130,
            facecolors="none",
            edgecolors="red",
            linewidths=1.6,
            zorder=4,
        )

    # P3 hull markers: red squares
    if mask_p3.any():
        axC.scatter(
            x_hull[mask_p3], c_hull[mask_p3],
            marker="s",
            s=130,
            facecolors="none",
            edgecolors="red",
            linewidths=1.6,
            zorder=4,
        )

    # O1 hull markers: red triangles (optional but consistent with voltage)
    if mask_o1.any():
        axC.scatter(
            x_hull[mask_o1], c_hull[mask_o1] * 3,
            marker="^",
            s=130,
            facecolors="none",
            edgecolors="red",
            linewidths=1.6,
            zorder=4,
        )

    # --- Style the c-axis to red (label, ticks, spine) ---
    axC.set_ylabel("c lattice parameter (Å)", color="red")
    axC.tick_params(axis="y", colors="red")
    axC.spines["right"].set_color("red")


    # -------------------------------------------------
    # Shade consecutive P3–P3 regions in the background
    # -------------------------------------------------
    if p3_spans:
        # Sort and merge overlapping/adjacent spans so shading looks continuous
        p3_spans_sorted = sorted(p3_spans, key=lambda s: s[0])
        merged_spans = []
        cur_start, cur_end = p3_spans_sorted[0]

        eps = 1e-6  # small tolerance for floating point

        for (x_start, x_end) in p3_spans_sorted[1:]:
            if x_start <= cur_end + eps:
                # overlap or touch → extend current span
                cur_end = max(cur_end, x_end)
            else:
                merged_spans.append((cur_start, cur_end))
                cur_start, cur_end = x_start, x_end

        merged_spans.append((cur_start, cur_end))

        # Now shade each merged span with a bright red background
        # Use a strong red but high transparency so lines/markers stay visible.
        for (x_start, x_end) in merged_spans:
            axV.axvspan(
                x_start,
                x_end,
                facecolor="red",
                alpha=0.12,      # bright but not overwhelming
                zorder=0,        # behind lines/markers
            )

    # -------------------------------------------------
    # Gradient background for O3–P3 plateaus:
    # white → red, with red matching the solid P3 color
    # -------------------------------------------------
    if gradient_spans:
        # Use final y-limits so gradient spans full vertical range
        y_min, y_max = axV.get_ylim()

        # colormap: 0 -> white, 1 -> red
        cmap_wr = LinearSegmentedColormap.from_list("white_red", ["white", "red"])

        for (x_start, x_end, p3_on_right) in gradient_spans:
            # horizontal gradient from 0..1 or 1..0
            grad = np.linspace(0, 1, 256).reshape(1, -1)  # shape (1, 256)
            if not p3_on_right:
                # P3 on left: red on left, white on right
                grad_to_plot = grad[:, ::-1]
            else:
                # P3 on right: white on left, red on right
                grad_to_plot = grad

            axV.imshow(
                grad_to_plot,
                extent=(x_start, x_end, y_min, y_max),
                origin="lower",
                aspect="auto",
                cmap=cmap_wr,
                alpha=0.36,   # match alpha with solid P3 red
                zorder=0,     # behind lines and markers
            )


    axV.grid(True, linestyle="--", alpha=0.35)
    axV.set_axisbelow(True)

    # Save voltage plot
    outV = Path(f"../data/figures/voltage_curve_DFT_{tm}.pdf")
    outV.parent.mkdir(parents=True, exist_ok=True)
    figV.savefig(outV, dpi=300, bbox_inches="tight")
    plt.close(figV)

out_paths


[]

In [10]:
# -----------------------------------------------------------------------------------------
# NEW CELL: Plot Theoretical + Experimental Curves Together
# -----------------------------------------------------------------------------------------

# Configuration for experimental data files
# We map the TM element to the specific CSV file and define the data type.

# Plotting convex hulls per TM
tms = sorted(df["tm"].unique())
# -----------------------------------------------------------------------------------------
# NEW CELL: Final Fix for NFO (Auto-Detection of Capacity vs Composition)
# -----------------------------------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

# 1. Configuration
experiment_map = {
    "Co": {
        "file": "AFM2016_NCO.csv",
        "label": "Exp (Kubota et al. AFM 2016)",
        "color": "blue",
        "is_capacity": True
    },
    "Fe": {
        "file": "AEM2018_NFO.csv",
        "label": "Exp (Kubota et al. AEM 2018)",
        "color": "blue",
        "is_capacity": False 
    },
    "Mn": {
        "file": "AFM2018_NMO_charge.csv",
        "label": "Exp (Chen et al. AFM 2018)",
        "color": "blue",
        "is_capacity": False 
    },
    "Ti": {
        "file": "EES2015_NTO.csv", 
        "label": "Exp (Wu et al. EES 2015)",
        "color": "blue",
        "is_capacity": True
    },
    "Ni": {
        "file": "ChemRxiv_NNO.csv", 
        "label": "Exp (Steele et al. ChemRxiv 2026)",
        "color": "blue",
        "is_capacity": False
    },
    "V": {
        "file": "IC2012_NVO.csv", 
        "label": "Exp (Didier et al. Inorg. Chem. 2012)",
        "color": "blue",
        "is_capacity": False
    },
    "Cr": {
        "file": "JPC2015_NCrO.csv", 
        "label": "Exp (Kubota et al. JPC 2015)",
        "color": "blue",
        "is_capacity": True
    }
}

def robust_load_csv(path):
    """Loads CSV, skipping metadata lines, returns DataFrame with cols [0, 1]."""
    clean_rows = []
    if not Path(path).exists():
        return pd.DataFrame()
        
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip metadata or empty lines
            if not line or line.startswith("[") or line.startswith("#") or "source" in line:
                continue
            # Split by comma or whitespace
            parts = line.split(',') if ',' in line else line.split()
            if len(parts) >= 2:
                try:
                    clean_rows.append((float(parts[0]), float(parts[1])))
                except ValueError:
                    continue
    return pd.DataFrame(clean_rows, columns=[0, 1])

out_dir = Path("../data/figures/include_exp")
out_dir.mkdir(parents=True, exist_ok=True)

print("Generating plots with Auto-Detection for Data Type...")

for tm in sorted(df["tm"].unique()):
    df_tm = df[df["tm"] == tm].copy()
    if df_tm.empty:
        continue

    # --- 1. THEORETICAL DATA ---
    df_pts = (
        df_tm
        .groupby(["ordering_short", "phase", "xNa"], as_index=False)
        .agg({"E_TMO2_eV": "mean", "group_label": "first"})
    )
    if df_pts.empty: continue

    xs_arr = df_pts["xNa"].to_numpy()
    E_arr = df_pts["E_TMO2_eV"].to_numpy()
    hull_idx = lower_hull_indices_scipy(xs_arr, E_arr)
    hull_idx_sorted = sorted(hull_idx, key=lambda i: xs_arr[i])
    
    x_hull = np.array([xs_arr[i] for i in hull_idx_sorted], dtype=float)
    E_hull = np.array([E_arr[i] for i in hull_idx_sorted], dtype=float)

    V_segments = []
    for i in range(len(x_hull) - 1):
        x1, x2 = x_hull[i], x_hull[i + 1]
        if np.isclose(x1, x2): continue
        mu = (E_hull[i + 1] - E_hull[i]) / (x2 - x1)
        V_segments.append((x1, x2, -(mu - E_Na_ref)))

    # --- 2. EXPERIMENTAL DATA LOADING & AUTO-DETECT ---
    x_exp_plot, v_exp_plot = [], []
    has_exp = False
    exp_label = ""
    exp_color = "blue"
    
    # Track min/max voltage for dynamic limits
    all_voltages = [v for _, _, v in V_segments]
    
    if tm in experiment_map:
        cfg = experiment_map[tm]
        df_exp = robust_load_csv(Path(f"../data/experiments/{cfg['file']}"))
        
        if not df_exp.empty:
            has_exp = True
            exp_label = cfg['label']
            exp_color = cfg['color']
            
            raw_x = df_exp[0].values
            raw_v = df_exp[1].values
            
            # --- AUTO DETECT LOGIC ---
            max_val = np.max(raw_x)
            C_theo = theoretical_capacity_full_mAh_g(tm)
            
            # If config says Capacity, but values are tiny (< 5.0), force Composition
            if cfg["is_capacity"] and max_val < 5.0:
                print(f"[{tm}] Note: Configured as Capacity, but data max={max_val:.2f}. Treating as Composition x.")
                x_converted = raw_x
            
            # If config says Composition, but values are huge (> 5.0), force Capacity
            elif (not cfg["is_capacity"]) and max_val > 5.0:
                 print(f"[{tm}] Note: Configured as Composition, but data max={max_val:.2f}. Treating as Capacity.")
                 x_converted = 1.0 - (raw_x / C_theo)
                 
            # Otherwise respect config
            elif cfg["is_capacity"]:
                x_converted = 1.0 - (raw_x / C_theo)
            else:
                x_converted = raw_x
                
            # Sort for clean line plotting
            combined = sorted(zip(x_converted, raw_v), key=lambda p: p[0])
            x_exp_plot = np.array([p[0] for p in combined])
            v_exp_plot = np.array([p[1] for p in combined])
            
            all_voltages.extend(v_exp_plot)

    # --- 3. DETERMINE AXIS LIMITS ---
    v_min = min(all_voltages) if all_voltages else 0.0
    v_max = max(all_voltages) if all_voltages else 4.5
    margin = 0.1 * (v_max - v_min) if v_max > v_min else 0.5
    y_limit_min = max(-0.5, v_min - margin)
    y_limit_max = v_max + margin

    # --- 4. PLOTTING ---
    figV, axV = plt.subplots(figsize=(6, 4.5))
    legend_elements = [Line2D([0], [0], color='black', lw=2, label='DFT (This Work)')]

    # Plot Theoretical
    for i, (x1, x2, V_seg) in enumerate(V_segments):
        idx_l, idx_r = hull_idx_sorted[i], hull_idx_sorted[i+1]
        kind_l = phase_kind(df_pts.loc[idx_l, "phase"])
        kind_r = phase_kind(df_pts.loc[idx_r, "phase"])
        
        style_l = P3_LINESTYLE if kind_l == "P3" else "solid"
        style_r = P3_LINESTYLE if kind_r == "P3" else "solid"
        
        x_mid = 0.5 * (x1 + x2)
        if (kind_l == "O3" and kind_r == "P3") or (kind_l == "P3" and kind_r == "O3"):
            axV.hlines(V_seg, x1, x_mid, lw=3, color="black", linestyles=style_l)
            axV.hlines(V_seg, x_mid, x2, lw=3, color="black", linestyles=style_r)
        else:
            s = P3_LINESTYLE if (kind_l=="P3" and kind_r=="P3") else "solid"
            axV.hlines(V_seg, x1, x2, lw=3, color="black", linestyles=s)
            
    # Vertical lines
    for j in range(1, len(x_hull) - 1):
        x_b = x_hull[j]
        v_l, v_r = V_segments[j-1][2], V_segments[j][2]
        ph = df_pts.loc[hull_idx_sorted[j], "phase"]
        s = P3_LINESTYLE if phase_kind(ph) == "P3" else "solid"
        axV.vlines(x_b, min(v_l, v_r), max(v_l, v_r), lw=3, color="black", linestyles=s)

    # Plot Experimental
    if has_exp:
        axV.plot(x_exp_plot, v_exp_plot, color=exp_color, linestyle="--", lw=2, alpha=0.8)
        legend_elements.append(Line2D([0], [0], color=exp_color, ls='--', lw=2, label=exp_label))

    # Labels & Limits
    axV.set_xlabel(f"x in Na$_x${tm}O$_2$")
    axV.set_ylabel("Voltage vs Na/Na$^+$ (V)")
    axV.set_xlim(0, 1)
    axV.set_ylim(y_limit_min, y_limit_max)
    axV.grid(True, linestyle="--", alpha=0.35)
    axV.legend(
        handles=legend_elements, 
        loc="best", 
        frameon=True, 
        fontsize=10,        # Adjust text size
        fancybox=True,      # Rounded corners
        shadow=True,        # Add a shadow effect
        framealpha=0.9,     # Slight transparency (0=transparent, 1=opaque)
        edgecolor="black"   # distinct border color
    )

    # --- 5. SMART TOP AXIS (CAPACITY) ---
    axTop = axV.twiny()
    x_min, x_max = axV.get_xlim()
    axTop.set_xlim(x_max, x_min) # Inverted x-axis
    axTop.set_xlabel("Theoretical capacity (mAh/g)")

    C_full = theoretical_capacity_full_mAh_g(tm)
    step = 50.0
    max_cap = float(C_full)
    
    # Smart filtering for ticks
    collision_threshold = 20.0 
    regular_ticks = np.arange(0.0, max_cap + 0.1, step)
    
    final_ticks = []
    for t in regular_ticks:
        if abs(t - max_cap) > collision_threshold:
            final_ticks.append(t)
    final_ticks.append(max_cap)
    final_ticks = sorted(set(final_ticks))
    
    # Map ticks to x positions (0..1 range)
    # x = 1 - Cap/C_full
    tick_pos_in_x_units = [1.0 - (cap / C_full) for cap in final_ticks]
    
    # Apply to top axis
    axTop.set_xlim(axV.get_xlim()) 
    axTop.set_xticks(tick_pos_in_x_units)
    axTop.set_xticklabels([f"{t:.0f}" for t in final_ticks])
    
    save_path = out_dir / f"voltage_curve_fixed_{tm}.pdf"
    figV.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(figV)
    print(f"Saved: {save_path}")

print("Done.")


Generating plots with Auto-Detection for Data Type...
Saved: ../data/figures/include_exp/voltage_curve_fixed_Co.pdf
Saved: ../data/figures/include_exp/voltage_curve_fixed_Cr.pdf
Saved: ../data/figures/include_exp/voltage_curve_fixed_Fe.pdf
Saved: ../data/figures/include_exp/voltage_curve_fixed_Mn.pdf
Saved: ../data/figures/include_exp/voltage_curve_fixed_Ni.pdf
Saved: ../data/figures/include_exp/voltage_curve_fixed_Ti.pdf
Saved: ../data/figures/include_exp/voltage_curve_fixed_V.pdf
Done.


In [ ]:
# Styled voltage plots with P3-region background (singleTM, synced with doubleTM styling)
if "df" not in globals():
    raise RuntimeError("Run the setup/loading cells first so `df` and helper functions are available.")

if "experiment_map" not in globals():
    experiment_map = {
        "Co": {"file": "AFM2016_NCO.csv", "label": "Exp (Kubota et al. AFM 2016)", "is_capacity": True},
        "Fe": {"file": "AEM2018_NFO.csv", "label": "Exp (Kubota et al. AEM 2018)", "is_capacity": False},
        "Mn": {"file": "AFM2018_NMO_charge.csv", "label": "Exp (Chen et al. AFM 2018)", "is_capacity": False},
        "Ti": {"file": "EES2015_NTO.csv", "label": "Exp (Wu et al. EES 2015)", "is_capacity": True},
        "Ni": {"file": "ChemRxiv_NNO.csv", "label": "Exp (Steele et al. ChemRxiv 2026)", "is_capacity": False},
        "V": {"file": "IC2012_NVO.csv", "label": "Exp (Didier et al. Inorg. Chem. 2012)", "is_capacity": False},
        "Cr": {"file": "JPC2015_NCrO.csv", "label": "Exp (Kubota et al. JPC 2015)", "is_capacity": True},
    }

if "robust_load_csv" not in globals():
    def robust_load_csv(path):
        clean_rows = []
        p = Path(path)
        if not p.exists():
            return pd.DataFrame()
        with p.open("r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = [x.strip() for x in line.split(",")]
                if len(parts) < 2:
                    continue
                try:
                    x = float(parts[0])
                    v = float(parts[1])
                except ValueError:
                    continue
                clean_rows.append((x, v))
        return pd.DataFrame(clean_rows)

tms = sorted(df["tm"].unique())
out_paths_p3_styled = []

for tm in tms:
    df_tm = df[df["tm"] == tm].copy()
    if df_tm.empty:
        continue

    df_pts = (
        df_tm
        .groupby(["ordering_short", "phase", "xNa"], as_index=False)
        .agg({"E_TMO2_eV": "mean", "lattice_c_A": "mean"})
    )
    if len(df_pts) < 2:
        continue

    xs = df_pts["xNa"].to_numpy(dtype=float)
    ys = df_pts["E_TMO2_eV"].to_numpy(dtype=float)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    if len(hull_idx) < 2:
        continue

    hull_idx = sorted(hull_idx, key=lambda i: xs[i])
    x_hull = np.array([xs[i] for i in hull_idx], dtype=float)
    e_hull = np.array([ys[i] for i in hull_idx], dtype=float)
    c_hull = np.array([df_pts.iloc[i]["lattice_c_A"] for i in hull_idx], dtype=float)
    phase_hull = [str(df_pts.iloc[i]["phase"]) for i in hull_idx]
    c_plot = c_hull.copy()
    o1_mask = np.array([phase_kind(ph) == "O1" for ph in phase_hull], dtype=bool)
    c_plot[o1_mask] = 3.0 * c_plot[o1_mask]

    V_segments = []
    phase_regions = {"O3": [], "P3": [], "O1": []}
    for i in range(len(x_hull) - 1):
        x1, x2 = x_hull[i], x_hull[i + 1]
        if np.isclose(x2, x1):
            continue
        mu_na = (e_hull[i + 1] - e_hull[i]) / (x2 - x1)
        V_seg = -(mu_na - E_Na_ref)
        V_segments.append((x1, x2, V_seg))

        kind_left = phase_kind(phase_hull[i])
        kind_right = phase_kind(phase_hull[i + 1])

        if kind_left == kind_right and kind_left in phase_regions:
            phase_regions[kind_left].append((x1, x2))
            continue

        edge = min(PHASE_CHANGE_EDGE_WIDTH, 0.5 * (x2 - x1))
        if kind_left in phase_regions and edge > 0:
            phase_regions[kind_left].append((x1, x1 + edge))
        if kind_right in phase_regions and edge > 0:
            phase_regions[kind_right].append((x2 - edge, x2))

    if not V_segments:
        continue

    has_exp = False
    x_exp_plot, v_exp_plot = [], []
    exp_label = "Experimental"
    if "experiment_map" in globals() and "robust_load_csv" in globals() and tm in experiment_map:
        cfg = experiment_map[tm]
        df_exp = robust_load_csv(Path(f"../data/experiments/{cfg['file']}"))
        if not df_exp.empty:
            raw_x = df_exp[0].values
            raw_v = df_exp[1].values
            C_theo = theoretical_capacity_full_mAh_g(tm)
            max_val = np.max(raw_x) if len(raw_x) else 0.0
            if cfg["is_capacity"] and max_val < 5.0:
                x_converted = raw_x
            elif (not cfg["is_capacity"]) and max_val > 5.0:
                x_converted = 1.0 - (raw_x / C_theo)
            elif cfg["is_capacity"]:
                x_converted = 1.0 - (raw_x / C_theo)
            else:
                x_converted = raw_x
            combined = sorted(zip(x_converted, raw_v), key=lambda p: p[0])
            if combined:
                x_exp_plot = np.array([p[0] for p in combined])
                v_exp_plot = np.array([p[1] for p in combined])
                has_exp = True
                exp_label = cfg.get("label", "Experimental")

    fig, ax = plt.subplots(figsize=(6, 4.5))

    phase_colors = {"O3": "#ff4d4f", "P3": "#1d8cff", "O1": "#22c55e"}
    eps = 1e-6
    for phase_kind_target, regions in phase_regions.items():
        if not regions:
            continue
        regions_sorted = sorted(regions, key=lambda seg: seg[0])
        merged_spans = []
        cur_start, cur_end = regions_sorted[0]
        for x_start, x_end in regions_sorted[1:]:
            if x_start <= cur_end + eps:
                cur_end = max(cur_end, x_end)
            else:
                merged_spans.append((cur_start, cur_end))
                cur_start, cur_end = x_start, x_end
        merged_spans.append((cur_start, cur_end))
        for x_start, x_end in merged_spans:
            ax.axvspan(x_start, x_end, facecolor=phase_colors[phase_kind_target], alpha=0.20, zorder=0)

    x_plot = []
    v_plot = []
    for x1, x2, v in V_segments:
        x_plot.extend([x1, x2])
        v_plot.extend([v, v])

    ax.plot(x_plot, v_plot, color="black", linewidth=2.5, label="DFT (Tight Conv.)")

    ax_c = ax.twinx()
    valid_c = np.isfinite(c_plot)
    if np.any(valid_c):
        ax_c.plot(
            x_hull[valid_c],
            c_plot[valid_c],
            color="tab:gray",
            marker="s",
            markersize=10.0,
            markeredgecolor="black",
            markeredgewidth=1.4,
            linewidth=1.8,
            linestyle=(0, (3.5, 2.2)),
            alpha=0.9,
            label="Stable c-axis",
        )
    ax_c.set_ylabel("C-axis length (Å)", fontsize=20)
    ax_c.set_ylim(13, 18)
    ax_c.tick_params(axis="y", labelsize=18)

    if has_exp:
        ax.plot(x_exp_plot, v_exp_plot, color="tab:red", linestyle=(0, (3, 2)), linewidth=2.2, alpha=0.95, label=exp_label)

    ax.text(0.02, 0.58, r"$\bf{Red:\ O3\ stable}$", transform=ax.transAxes, ha="left", va="center", fontsize=10, color="#ff4d4f")
    ax.text(0.02, 0.52, r"$\bf{Blue:\ P3\ stable}$", transform=ax.transAxes, ha="left", va="center", fontsize=10, color="#1d8cff")
    ax.text(0.02, 0.46, r"$\bf{Green:\ O1\ stable}$", transform=ax.transAxes, ha="left", va="center", fontsize=10, color="#22c55e")

    ax.set_xlabel(f"x in Na$_{{\mathrm{{x}}}}${tm}O$_2$", fontsize=20)
    ax.set_ylabel("Voltage (V vs Na/Na$^+$)", fontsize=20)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0, 6)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.tick_params(axis="both", labelsize=18)

    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_c.get_legend_handles_labels()
    legend_loc = "center right" if tm == "Ti" else "lower right"
    ax.legend(
        handles_l + handles_r,
        labels_l + labels_r,
        loc=legend_loc,
        fontsize=11,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.78,
        edgecolor="black",
        facecolor="white",
    )

    ax_top = ax.twiny()
    x_min, x_max = ax.get_xlim()
    ax_top.set_xlim(x_max, x_min)
    C_full = theoretical_capacity_full_mAh_g(tm)
    ax_top.set_xlabel("Capacity (mAh/g)", fontsize=20)
    ticks = np.arange(0, C_full + 0.1, 50)
    if len(ticks) > 1 and abs(ticks[-1] - C_full) < 10:
        ticks = ticks[:-1]
    ticks = list(ticks) + [C_full]
    ticks = [t for t in ticks if abs(t - 250.0) > 1e-9]
    ax_top.set_xticks([1.0 - (t / C_full) for t in ticks])
    ax_top.set_xticklabels([f"{t:.0f}" for t in ticks])
    ax_top.tick_params(axis="x", labelsize=18)

    outpath = Path(f"../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_{tm}.pdf")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    out_paths_p3_styled.append(str(outpath))

print(f"Saved {len(out_paths_p3_styled)} styled voltage plots")
out_paths_p3_styled





Saved 7 styled voltage plots


['../data/figures/include_exp/voltage_curve_DFT_tighterconv_p3styled_Co.pdf',
 '../data/figures/include_exp/voltage_curve_DFT_tighterconv_p3styled_Cr.pdf',
 '../data/figures/include_exp/voltage_curve_DFT_tighterconv_p3styled_Fe.pdf',
 '../data/figures/include_exp/voltage_curve_DFT_tighterconv_p3styled_Mn.pdf',
 '../data/figures/include_exp/voltage_curve_DFT_tighterconv_p3styled_Ni.pdf',
 '../data/figures/include_exp/voltage_curve_DFT_tighterconv_p3styled_Ti.pdf',
 '../data/figures/include_exp/voltage_curve_DFT_tighterconv_p3styled_V.pdf']

In [13]:
# Styled voltage plots with P3-region background (singleTM, synced with doubleTM styling)
if "df" not in globals():
    raise RuntimeError("Run the setup/loading cells first so `df` and helper functions are available.")

if "experiment_map" not in globals():
    experiment_map = {
        "Co": {"file": "AFM2016_NCO.csv", "label": "Exp (Kubota et al. AFM 2016)", "is_capacity": True},
        "Fe": {"file": "AEM2018_NFO.csv", "label": "Exp (Kubota et al. AEM 2018)", "is_capacity": False},
        "Mn": {"file": "AFM2018_NMO_charge.csv", "label": "Exp (Chen et al. AFM 2018)", "is_capacity": False},
        "Ti": {"file": "EES2015_NTO.csv", "label": "Exp (Wu et al. EES 2015)", "is_capacity": True},
        "Ni": {"file": "ChemRxiv_NNO.csv", "label": "Exp (Steele et al. ChemRxiv 2026)", "is_capacity": False},
        "V": {"file": "IC2012_NVO.csv", "label": "Exp (Didier et al. Inorg. Chem. 2012)", "is_capacity": False},
        "Cr": {"file": "JPC2015_NCrO.csv", "label": "Exp (Kubota et al. JPC 2015)", "is_capacity": True},
    }

if "robust_load_csv" not in globals():
    def robust_load_csv(path):
        clean_rows = []
        p = Path(path)
        if not p.exists():
            return pd.DataFrame()
        with p.open("r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = [x.strip() for x in line.split(",")]
                if len(parts) < 2:
                    continue
                try:
                    x = float(parts[0])
                    v = float(parts[1])
                except ValueError:
                    continue
                clean_rows.append((x, v))
        return pd.DataFrame(clean_rows)

tms = sorted(df["tm"].unique())
out_paths_p3_styled = []

for tm in tms:
    df_tm = df[df["tm"] == tm].copy()
    if df_tm.empty:
        continue

    df_pts = (
        df_tm
        .groupby(["ordering_short", "phase", "xNa"], as_index=False)
        .agg({"E_TMO2_eV": "mean", "lattice_c_A": "mean"})
    )
    if len(df_pts) < 2:
        continue

    xs = df_pts["xNa"].to_numpy(dtype=float)
    ys = df_pts["E_TMO2_eV"].to_numpy(dtype=float)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    if len(hull_idx) < 2:
        continue

    hull_idx = sorted(hull_idx, key=lambda i: xs[i])
    x_hull = np.array([xs[i] for i in hull_idx], dtype=float)
    e_hull = np.array([ys[i] for i in hull_idx], dtype=float)
    c_hull = np.array([df_pts.iloc[i]["lattice_c_A"] for i in hull_idx], dtype=float)
    phase_hull = [str(df_pts.iloc[i]["phase"]) for i in hull_idx]
    c_plot = c_hull.copy()
    o1_mask = np.array([phase_kind(ph) == "O1" for ph in phase_hull], dtype=bool)
    c_plot[o1_mask] = 3.0 * c_plot[o1_mask]

    V_segments = []
    phase_regions = {"O3": [], "P3": [], "O1": []}
    for i in range(len(x_hull) - 1):
        x1, x2 = x_hull[i], x_hull[i + 1]
        if np.isclose(x2, x1):
            continue
        mu_na = (e_hull[i + 1] - e_hull[i]) / (x2 - x1)
        V_seg = -(mu_na - E_Na_ref)
        V_segments.append((x1, x2, V_seg))

        kind_left = phase_kind(phase_hull[i])
        kind_right = phase_kind(phase_hull[i + 1])

        if kind_left == kind_right and kind_left in phase_regions:
            phase_regions[kind_left].append((x1, x2))
            continue

        edge = min(PHASE_CHANGE_EDGE_WIDTH, 0.5 * (x2 - x1))
        if kind_left in phase_regions and edge > 0:
            phase_regions[kind_left].append((x1, x1 + edge))
        if kind_right in phase_regions and edge > 0:
            phase_regions[kind_right].append((x2 - edge, x2))

    if not V_segments:
        continue

    has_exp = False
    x_exp_plot, v_exp_plot = [], []
    exp_label = "Experimental"
    if "experiment_map" in globals() and "robust_load_csv" in globals() and tm in experiment_map:
        cfg = experiment_map[tm]
        df_exp = robust_load_csv(Path(f"../data/experiments/{cfg['file']}"))
        if not df_exp.empty:
            raw_x = df_exp[0].values
            raw_v = df_exp[1].values
            C_theo = theoretical_capacity_full_mAh_g(tm)
            max_val = np.max(raw_x) if len(raw_x) else 0.0
            if cfg["is_capacity"] and max_val < 5.0:
                x_converted = raw_x
            elif (not cfg["is_capacity"]) and max_val > 5.0:
                x_converted = 1.0 - (raw_x / C_theo)
            elif cfg["is_capacity"]:
                x_converted = 1.0 - (raw_x / C_theo)
            else:
                x_converted = raw_x
            combined = sorted(zip(x_converted, raw_v), key=lambda p: p[0])
            if combined:
                x_exp_plot = np.array([p[0] for p in combined])
                v_exp_plot = np.array([p[1] for p in combined])
                has_exp = True
                exp_label = cfg.get("label", "Experimental")

    fig, ax = plt.subplots(figsize=(6, 4.5))

    phase_colors = {"O3": "#ff4d4f", "P3": "#1d8cff", "O1": "#22c55e"}
    eps = 1e-6
    for phase_kind_target, regions in phase_regions.items():
        if not regions:
            continue
        regions_sorted = sorted(regions, key=lambda seg: seg[0])
        merged_spans = []
        cur_start, cur_end = regions_sorted[0]
        for x_start, x_end in regions_sorted[1:]:
            if x_start <= cur_end + eps:
                cur_end = max(cur_end, x_end)
            else:
                merged_spans.append((cur_start, cur_end))
                cur_start, cur_end = x_start, x_end
        merged_spans.append((cur_start, cur_end))
        for x_start, x_end in merged_spans:
            ax.axvspan(x_start, x_end, facecolor=phase_colors[phase_kind_target], alpha=0.20, zorder=0)

    x_plot = []
    v_plot = []
    for x1, x2, v in V_segments:
        x_plot.extend([x1, x2])
        v_plot.extend([v, v])

    ax.plot(x_plot, v_plot, color="black", linewidth=2.5, label="DFT (Tight Conv.)")

    ax_c = ax.twinx()
    valid_c = np.isfinite(c_plot)
    if np.any(valid_c):
        ax_c.plot(
            x_hull[valid_c],
            c_plot[valid_c],
            color="tab:gray",
            marker="s",
            markersize=10.0,
            markeredgecolor="black",
            markeredgewidth=1.4,
            linewidth=1.8,
            linestyle=(0, (3.5, 2.2)),
            alpha=0.9,
            label="Stable c-axis",
        )
    ax_c.set_ylabel("C-axis length (Å)", fontsize=20)
    ax_c.set_ylim(13, 18)
    ax_c.tick_params(axis="y", labelsize=18)

    if has_exp:
        ax.plot(x_exp_plot, v_exp_plot, color="tab:red", linestyle=(0, (3, 2)), linewidth=2.2, alpha=0.95, label=exp_label)
    """
    ax.text(0.02, 0.58, r"$\bf{Red:\ O3\ stable}$", transform=ax.transAxes, ha="left", va="center", fontsize=10, color="#ff4d4f")
    ax.text(0.02, 0.52, r"$\bf{Blue:\ P3\ stable}$", transform=ax.transAxes, ha="left", va="center", fontsize=10, color="#1d8cff")
    ax.text(0.02, 0.46, r"$\bf{Green:\ O1\ stable}$", transform=ax.transAxes, ha="left", va="center", fontsize=10, color="#22c55e")
    """
    ax.set_xlabel(f"x in Na$_{{\mathrm{{x}}}}${tm}O$_2$", fontsize=20)
    ax.set_ylabel("Voltage (V vs Na/Na$^+$)", fontsize=20)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0, 6)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.tick_params(axis="both", labelsize=18)

    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_c.get_legend_handles_labels()
    legend_loc = "center right" if tm == "Ti" else "lower right"
    ax.legend(
        handles_l + handles_r,
        labels_l + labels_r,
        loc=legend_loc,
        fontsize=11,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.78,
        edgecolor="black",
        facecolor="white",
    )

    ax_top = ax.twiny()
    x_min, x_max = ax.get_xlim()
    ax_top.set_xlim(x_max, x_min)
    C_full = theoretical_capacity_full_mAh_g(tm)
    ax_top.set_xlabel("Capacity (mAh/g)", fontsize=20)
    ticks = np.arange(0, C_full + 0.1, 50)
    if len(ticks) > 1 and abs(ticks[-1] - C_full) < 10:
        ticks = ticks[:-1]
    ticks = list(ticks) + [C_full]
    ticks = [t for t in ticks if abs(t - 250.0) > 1e-9]
    ax_top.set_xticks([1.0 - (t / C_full) for t in ticks])
    ax_top.set_xticklabels([f"{t:.0f}" for t in ticks])
    ax_top.tick_params(axis="x", labelsize=18)

    outpath = Path(f"../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_{tm}.pdf")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    out_paths_p3_styled.append(str(outpath))

print(f"Saved {len(out_paths_p3_styled)} styled voltage plots")
out_paths_p3_styled





Saved 7 styled voltage plots


['../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_Co.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_Cr.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_Fe.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_Mn.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_Ni.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_Ti.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_p3styled_V.pdf']

In [12]:
# Voltage profile with Magnetic Moments.
import ast

if "df" not in globals():
    raise RuntimeError("Run the setup/loading cells first so `df` and helper functions are available.")

if "experiment_map" not in globals():
    experiment_map = {
        "Co": {"file": "AFM2016_NCO.csv", "label": "Exp (Kubota et al. AFM 2016)", "is_capacity": True},
        "Fe": {"file": "AEM2018_NFO.csv", "label": "Exp (Kubota et al. AEM 2018)", "is_capacity": False},
        "Mn": {"file": "AFM2018_NMO_charge.csv", "label": "Exp (Chen et al. AFM 2018)", "is_capacity": False},
        "Ti": {"file": "EES2015_NTO.csv", "label": "Exp (Wu et al. EES 2015)", "is_capacity": True},
        "Ni": {"file": "ChemRxiv_NNO.csv", "label": "Exp (Steele et al. ChemRxiv 2026)", "is_capacity": False},
        "V": {"file": "IC2012_NVO.csv", "label": "Exp (Didier et al. Inorg. Chem. 2012)", "is_capacity": False},
        "Cr": {"file": "JPC2015_NCrO.csv", "label": "Exp (Kubota et al. JPC 2015)", "is_capacity": True},
    }

if "robust_load_csv" not in globals():
    def robust_load_csv(path):
        clean_rows = []
        p = Path(path)
        if not p.exists():
            return pd.DataFrame()
        with p.open("r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = [x.strip() for x in line.split(",")]
                if len(parts) < 2:
                    continue
                try:
                    x = float(parts[0])
                    v = float(parts[1])
                except ValueError:
                    continue
                clean_rows.append((x, v))
        return pd.DataFrame(clean_rows)


def avg_abs_tm_magmom(tm_magmom_dict_str, tm_symbol):
    if pd.isna(tm_magmom_dict_str):
        return np.nan
    try:
        parsed = ast.literal_eval(tm_magmom_dict_str)
    except Exception:
        return np.nan

    values = []
    if isinstance(parsed, dict):
        if tm_symbol in parsed:
            values = parsed[tm_symbol]
        else:
            for v in parsed.values():
                if isinstance(v, (list, tuple, np.ndarray)):
                    values.extend(v)
                else:
                    values.append(v)
    elif isinstance(parsed, (list, tuple, np.ndarray)):
        values = list(parsed)
    else:
        return np.nan

    arr = np.array(values, dtype=float)
    if arr.size == 0:
        return np.nan
    return float(np.mean(np.abs(arr)))


# Load magmom table and build lookup columns
magmom_df = pd.read_csv("../data/singleTM_magmom.csv")
magmom_df["ordering_short"] = magmom_df["ordering"].astype(str).str.replace("_w_vac", "", regex=False)
magmom_df["avg_abs_magmom"] = magmom_df.apply(
    lambda r: avg_abs_tm_magmom(r["tm_magmom_dict"], r["tm"]),
    axis=1,
)


tms = sorted(df["tm"].unique())
out_paths_magmom = []
stable_point_tables = []

for tm in tms:
    df_tm = df[df["tm"] == tm].copy()
    if df_tm.empty:
        continue

    # Keep explicit (phase, ordering, tm) identity for stable-point matching to magmom table
    df_pts = (
        df_tm
        .groupby(["phase", "ordering", "ordering_short", "xNa", "tm"], as_index=False)
        .agg({"E_TMO2_eV": "mean"})
    )
    if len(df_pts) < 2:
        continue

    xs = df_pts["xNa"].to_numpy(dtype=float)
    ys = df_pts["E_TMO2_eV"].to_numpy(dtype=float)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    if len(hull_idx) < 2:
        continue

    hull_pts = df_pts.iloc[hull_idx].sort_values("xNa").reset_index(drop=True)
    x_hull = hull_pts["xNa"].to_numpy(dtype=float)
    e_hull = hull_pts["E_TMO2_eV"].to_numpy(dtype=float)
    phase_hull = hull_pts["phase"].astype(str).tolist()

    # Exact key match first: (phase, ordering, tm)
    hull_merged = hull_pts.merge(
        magmom_df[["phase", "ordering", "tm", "avg_abs_magmom", "ordering_short"]],
        on=["phase", "ordering", "tm"],
        how="left",
        suffixes=("", "_mm"),
    )

    # Fallback: (phase, ordering_short, tm), if exact ordering key is missing
    missing_mask = hull_merged["avg_abs_magmom"].isna()
    if missing_mask.any():
        fallback = hull_merged.loc[missing_mask, ["phase", "ordering_short", "tm"]].merge(
            magmom_df[["phase", "ordering_short", "tm", "avg_abs_magmom"]],
            on=["phase", "ordering_short", "tm"],
            how="left",
        )
        hull_merged.loc[missing_mask, "avg_abs_magmom"] = fallback["avg_abs_magmom"].to_numpy()

    magmom_hull = hull_merged["avg_abs_magmom"].to_numpy(dtype=float)

    # Store stable-point info for checking
    stable_tbl = hull_merged[["tm", "phase", "ordering", "xNa", "avg_abs_magmom"]].copy()
    stable_tbl["tm"] = tm
    stable_point_tables.append(stable_tbl)

    V_segments = []
    phase_regions = {"O3": [], "P3": [], "O1": []}
    for i in range(len(x_hull) - 1):
        x1, x2 = x_hull[i], x_hull[i + 1]
        if np.isclose(x2, x1):
            continue
        mu_na = (e_hull[i + 1] - e_hull[i]) / (x2 - x1)
        V_seg = -(mu_na - E_Na_ref)
        V_segments.append((x1, x2, V_seg))

        kind_left = phase_kind(phase_hull[i])
        kind_right = phase_kind(phase_hull[i + 1])

        if kind_left == kind_right and kind_left in phase_regions:
            phase_regions[kind_left].append((x1, x2))
            continue

        edge = min(PHASE_CHANGE_EDGE_WIDTH, 0.5 * (x2 - x1))
        if kind_left in phase_regions and edge > 0:
            phase_regions[kind_left].append((x1, x1 + edge))
        if kind_right in phase_regions and edge > 0:
            phase_regions[kind_right].append((x2 - edge, x2))

    if not V_segments:
        continue

    has_exp = False
    x_exp_plot, v_exp_plot = [], []
    exp_label = "Experimental"
    if tm in experiment_map:
        cfg = experiment_map[tm]
        df_exp = robust_load_csv(Path(f"../data/experiments/{cfg['file']}"))
        if not df_exp.empty:
            raw_x = df_exp[0].values
            raw_v = df_exp[1].values
            C_theo = theoretical_capacity_full_mAh_g(tm)
            max_val = np.max(raw_x) if len(raw_x) else 0.0
            if cfg["is_capacity"] and max_val < 5.0:
                x_converted = raw_x
            elif (not cfg["is_capacity"]) and max_val > 5.0:
                x_converted = 1.0 - (raw_x / C_theo)
            elif cfg["is_capacity"]:
                x_converted = 1.0 - (raw_x / C_theo)
            else:
                x_converted = raw_x
            combined = sorted(zip(x_converted, raw_v), key=lambda p: p[0])
            if combined:
                x_exp_plot = np.array([p[0] for p in combined])
                v_exp_plot = np.array([p[1] for p in combined])
                has_exp = True
                exp_label = cfg.get("label", "Experimental")

    fig, ax = plt.subplots(figsize=(6, 4.5))

    phase_colors = {"O3": "#ff4d4f", "P3": "#1d8cff", "O1": "#22c55e"}
    eps = 1e-6
    for phase_kind_target, regions in phase_regions.items():
        if not regions:
            continue
        regions_sorted = sorted(regions, key=lambda seg: seg[0])
        merged_spans = []
        cur_start, cur_end = regions_sorted[0]
        for x_start, x_end in regions_sorted[1:]:
            if x_start <= cur_end + eps:
                cur_end = max(cur_end, x_end)
            else:
                merged_spans.append((cur_start, cur_end))
                cur_start, cur_end = x_start, x_end
        merged_spans.append((cur_start, cur_end))
        for x_start, x_end in merged_spans:
            ax.axvspan(x_start, x_end, facecolor=phase_colors[phase_kind_target], alpha=0.20, zorder=0)

    x_plot = []
    v_plot = []
    for x1, x2, v in V_segments:
        x_plot.extend([x1, x2])
        v_plot.extend([v, v])

    ax.plot(x_plot, v_plot, color="black", linewidth=2.5, label="DFT (Tight Conv.)")

    ax_mm = ax.twinx()
    valid_mm = np.isfinite(magmom_hull)
    if np.any(valid_mm):
        ax_mm.plot(
            x_hull[valid_mm],
            magmom_hull[valid_mm],
            color="#0b3d91",
            marker="D",
            markersize=10.0,
            markeredgecolor="black",
            markeredgewidth=1.0,
            linewidth=1.6,
            linestyle=(0, (1.2, 2.2)),
            alpha=0.95,
            label="Avg of Magnetic Moment",
            zorder=3,
        )
    ax_mm.set_ylabel(r"Avg of Magnetic Moment ($\mu_B$)", fontsize=20)
    ax_mm.set_ylim(0, 5)
    ax_mm.tick_params(axis="y", labelsize=18)

    if has_exp:
        ax.plot(x_exp_plot, v_exp_plot, color="tab:red", linestyle=(0, (3, 2)), linewidth=2.2, alpha=0.95, label=exp_label)

    ax.set_xlabel(f"x in Na$_{{\\mathrm{{x}}}}${tm}O$_2$", fontsize=20)
    ax.set_ylabel("Voltage (V vs Na/Na$^+$)", fontsize=20)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0, 6)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.tick_params(axis="both", labelsize=18)

    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_mm.get_legend_handles_labels()
    legend_loc = "upper right" if tm in {"Ti", "V", "Ni", "Co"} else "lower right"
    ax.legend(
        handles_l + handles_r,
        labels_l + labels_r,
        loc=legend_loc,
        fontsize=11,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.78,
        edgecolor="black",
        facecolor="white",
    )

    ax_top = ax.twiny()
    x_min, x_max = ax.get_xlim()
    ax_top.set_xlim(x_max, x_min)
    C_full = theoretical_capacity_full_mAh_g(tm)
    ax_top.set_xlabel("Capacity (mAh/g)", fontsize=20)
    ticks = np.arange(0, C_full + 0.1, 50)
    if len(ticks) > 1 and abs(ticks[-1] - C_full) < 10:
        ticks = ticks[:-1]
    ticks = list(ticks) + [C_full]
    ticks = [t for t in ticks if abs(t - 250.0) > 1e-9]
    ax_top.set_xticks([1.0 - (t / C_full) for t in ticks])
    ax_top.set_xticklabels([f"{t:.0f}" for t in ticks])
    ax_top.tick_params(axis="x", labelsize=18)

    outpath = Path(f"../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_{tm}.pdf")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    out_paths_magmom.append(str(outpath))

stable_points_magmom_df = pd.concat(stable_point_tables, ignore_index=True)
display(stable_points_magmom_df.sort_values(["tm", "xNa"]).reset_index(drop=True))

missing = stable_points_magmom_df[stable_points_magmom_df["avg_abs_magmom"].isna()]
if not missing.empty:
    print("Warning: missing magnetic moments for the following stable points:")
    display(missing.sort_values(["tm", "xNa"]).reset_index(drop=True))

print(f"Saved {len(out_paths_magmom)} voltage+magmom plots")
out_paths_magmom



,tm,phase,ordering,xNa,avg_abs_magmom
0,Co,O1_orderings,empty,0.000000,1.165000
1,Co,O3_orderings,AJT_b_w_vac,0.250000,0.843750
2,Co,P3_orderings,JLK_zeta2_w_vac,0.400000,0.667600
3,Co,P3_orderings,JLK_zeta3_w_vac,0.428571,0.634571
4,Co,P3_orderings,JLK_zeta_w_vac,0.500000,0.553000
5,Co,P3_orderings,JMAS_d,0.666667,0.366333
6,Co,O3_orderings,JMAS_e,1.000000,0.000000
7,Cr,O3_orderings,empty,0.000000,2.431667
8,Cr,O3_orderings,JLK_b_w_vac,0.375000,2.462750
9,Cr,O3_orderings,AJT_g_w_vac,0.500000,2.562500


Saved 7 voltage+magmom plots


['../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_Co.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_Cr.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_Fe.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_Mn.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_Ni.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_Ti.pdf',
 '../data/figures/singleTM/voltage_curve_DFT_tighterconv_magmom_V.pdf']